# 02 — Exploratory Data Analysis
### Credit Card Customer Intelligence & Churn Analytics

Answers business questions across four areas: customer demographics, spending behavior, customer loyalty, and — the core of this project — churn analysis. Every plot here is saved to `images/plots/` for reuse in the dashboard and executive report.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
PLOT_DIR = '../images/plots'

df = pd.read_csv('../data/processed/bankchurners_clean.csv')
df['Income_Category'] = pd.Categorical(df['Income_Category'],
    categories=['Less than $40K','$40K - $60K','$60K - $80K','$80K - $120K','$120K +','Unknown'], ordered=True)
df['Education_Level'] = pd.Categorical(df['Education_Level'],
    categories=['Uneducated','High School','College','Graduate','Post-Graduate','Doctorate','Unknown'], ordered=True)
df['Card_Category'] = pd.Categorical(df['Card_Category'], categories=['Blue','Silver','Gold','Platinum'], ordered=True)
print(df.shape)

(10127, 27)


## Section A — Customer Demographics

In [2]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0,0].hist(df['Customer_Age'], bins=25, color='#4C72B0', edgecolor='white')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Customer Age')

df['Gender'].value_counts().plot(kind='bar', ax=axes[0,1], color=['#4C72B0','#DD8452'])
axes[0,1].set_title('Gender Distribution')
axes[0,1].tick_params(axis='x', rotation=0)

df['Income_Category'].value_counts().sort_index().plot(kind='barh', ax=axes[1,0], color='#55A868')
axes[1,0].set_title('Income Distribution')

df['Education_Level'].value_counts().sort_index().plot(kind='barh', ax=axes[1,1], color='#C44E52')
axes[1,1].set_title('Education Level Distribution')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/demographics_overview.png', bbox_inches='tight')
plt.show()

In [3]:
print("Married vs Unmarried:")
print(df['Marital_Status'].value_counts())
print()
print("Card Category split:")
print(df['Card_Category'].value_counts())

Married vs Unmarried:
Marital_Status
Married     4687
Single      3943
Unknown      749
Divorced     748
Name: count, dtype: int64

Card Category split:
Card_Category
Blue        9436
Silver       555
Gold         116
Platinum      20
Name: count, dtype: int64


**Reading:** Customer base skews mid-40s in age, roughly balanced across gender, concentrated in lower income bands (Less than $40K is the single largest group), Graduate is the most common education level, and the overwhelming majority hold the entry-level **Blue** card — Platinum is a rounding error at ~0.2% of customers.

## Section B — Spending Behavior

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0,0].hist(df['Total_Trans_Ct'], bins=30, color='#4C72B0', edgecolor='white')
axes[0,0].set_title('Transaction Count Distribution')

axes[0,1].hist(df['Total_Trans_Amt'], bins=30, color='#DD8452', edgecolor='white')
axes[0,1].set_title('Transaction Amount Distribution')

axes[1,0].hist(df['Credit_Limit'], bins=30, color='#55A868', edgecolor='white')
axes[1,0].set_title('Credit Limit Distribution')

axes[1,1].hist(df['Avg_Utilization_Ratio'], bins=30, color='#C44E52', edgecolor='white')
axes[1,1].set_title('Utilization Ratio Distribution')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/spending_behavior.png', bbox_inches='tight')
plt.show()

**Reading:** Transaction count is bimodal — there's a distinct low-activity cluster below ~50 transactions and a higher-activity cluster above it, worth revisiting in segmentation. Credit limit is heavily right-skewed (most customers cluster at lower limits, with a long tail of high-limit customers). Utilization is skewed toward the low end — most customers carry a small revolving balance relative to their limit.

## Section C — Customer Loyalty

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].hist(df['Months_on_book'], bins=20, color='#4C72B0', edgecolor='white')
axes[0].set_title('Months on Book (Tenure)')

df['Months_Inactive_12_mon'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#DD8452')
axes[1].set_title('Inactive Months (last 12mo)')
axes[1].tick_params(axis='x', rotation=0)

df['Total_Relationship_Count'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='#55A868')
axes[2].set_title('Products Held (Relationship Count)')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/customer_loyalty.png', bbox_inches='tight')
plt.show()

## Section D — Churn Analysis

The core of the project. Each plot below maps directly to one of the business questions in the brief.

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Churn vs Income
pd.crosstab(df['Income_Category'], df['Attrition_Flag'], normalize='index')['Attrited Customer']\
    .sort_index().plot(kind='bar', ax=axes[0,0], color='#C44E52')
axes[0,0].set_title('Churn Rate by Income Category')
axes[0,0].set_ylabel('Churn Rate')
axes[0,0].tick_params(axis='x', rotation=45)

# Churn vs Card Type
pd.crosstab(df['Card_Category'], df['Attrition_Flag'], normalize='index')['Attrited Customer']\
    .sort_index().plot(kind='bar', ax=axes[0,1], color='#C44E52')
axes[0,1].set_title('Churn Rate by Card Category')
axes[0,1].set_ylabel('Churn Rate')
axes[0,1].tick_params(axis='x', rotation=0)

# Churn vs Gender
pd.crosstab(df['Gender'], df['Attrition_Flag'], normalize='index')['Attrited Customer']\
    .plot(kind='bar', ax=axes[1,0], color='#C44E52')
axes[1,0].set_title('Churn Rate by Gender')
axes[1,0].set_ylabel('Churn Rate')
axes[1,0].tick_params(axis='x', rotation=0)

# Churn vs Age bucket
age_bins = pd.cut(df['Customer_Age'], bins=[0,30,40,50,60,100], labels=['<30','30-40','40-50','50-60','60+'])
pd.crosstab(age_bins, df['Attrition_Flag'], normalize='index')['Attrited Customer']\
    .plot(kind='bar', ax=axes[1,1], color='#C44E52')
axes[1,1].set_title('Churn Rate by Age Group')
axes[1,1].set_ylabel('Churn Rate')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/churn_demographics.png', bbox_inches='tight')
plt.show()

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Churn vs Credit Limit (boxplot)
sns.boxplot(data=df, x='Attrition_Flag', y='Credit_Limit', ax=axes[0,0], hue='Attrition_Flag',
            palette=['#55A868','#C44E52'], legend=False)
axes[0,0].set_title('Credit Limit by Churn Status')

# Churn vs Utilization (boxplot)
sns.boxplot(data=df, x='Attrition_Flag', y='Avg_Utilization_Ratio', ax=axes[0,1], hue='Attrition_Flag',
            palette=['#55A868','#C44E52'], legend=False)
axes[0,1].set_title('Utilization Ratio by Churn Status')

# Churn vs Transaction Count (boxplot)
sns.boxplot(data=df, x='Attrition_Flag', y='Total_Trans_Ct', ax=axes[1,0], hue='Attrition_Flag',
            palette=['#55A868','#C44E52'], legend=False)
axes[1,0].set_title('Transaction Count by Churn Status')

# Churn vs Months Inactive
pd.crosstab(df['Months_Inactive_12_mon'], df['Attrition_Flag'], normalize='index')['Attrited Customer']\
    .plot(kind='bar', ax=axes[1,1], color='#C44E52')
axes[1,1].set_title('Churn Rate by Months Inactive')
axes[1,1].set_ylabel('Churn Rate')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/churn_behavior.png', bbox_inches='tight')
plt.show()

In [8]:
# Churn vs Customer Tenure (Months on Book)
fig, ax = plt.subplots(figsize=(9, 5))
tenure_bins = pd.cut(df['Months_on_book'], bins=[0,24,36,48,60], labels=['<=24','25-36','37-48','49-60'])
pd.crosstab(tenure_bins, df['Attrition_Flag'], normalize='index')['Attrited Customer'].plot(kind='bar', color='#C44E52', ax=ax)
ax.set_title('Churn Rate by Customer Tenure')
ax.set_ylabel('Churn Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/churn_by_tenure.png', bbox_inches='tight')
plt.show()

**Key churn findings:**
- **Income** shows little separation — churn rate is fairly flat across income bands, suggesting income alone isn't a strong churn driver here
- **Card category** shows Platinum/Gold churning at a somewhat different rate than Blue, but sample sizes for premium cards are tiny (~20-100 customers), so this should be read cautiously
- **Gender and age** show only modest differences — not the primary drivers
- **Utilization and transaction activity** show the clearest separation: churned customers cluster at noticeably lower utilization and transaction counts
- **Months inactive** shows a fairly clean upward trend — more inactive months tracks with higher churn, supporting the "proactive engagement" recommendation
- **Tenure** shows churn is not concentrated only in new customers — even customers with 40+ months on book churn, which argues against a "just retain new customers longer" strategy

## Section E — Correlation Analysis

In [9]:
numeric_cols = ['Customer_Age','Dependent_count','Months_on_book','Total_Relationship_Count',
                'Months_Inactive_12_mon','Contacts_Count_12_mon','Credit_Limit','Total_Revolving_Bal',
                'Avg_Open_To_Buy','Total_Amt_Chng_Q4_Q1','Total_Trans_Amt','Total_Trans_Ct',
                'Total_Ct_Chng_Q4_Q1','Avg_Utilization_Ratio','Churn_Flag','Engagement_Score']

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, annot_kws={'size':7})
ax.set_title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [10]:
print("Strongest correlations with Churn_Flag:")
print(corr['Churn_Flag'].drop('Churn_Flag').sort_values(key=abs, ascending=False))

Strongest correlations with Churn_Flag:
Engagement_Score           -0.381377
Total_Trans_Ct             -0.371403
Total_Ct_Chng_Q4_Q1        -0.290054
Total_Revolving_Bal        -0.263053
Contacts_Count_12_mon       0.204491
Avg_Utilization_Ratio      -0.178410
Total_Trans_Amt            -0.168598
Months_Inactive_12_mon      0.152449
Total_Relationship_Count   -0.150005
Total_Amt_Chng_Q4_Q1       -0.131063
Credit_Limit               -0.023873
Dependent_count             0.018991
Customer_Age                0.018203
Months_on_book              0.013687
Avg_Open_To_Buy            -0.000285
Name: Churn_Flag, dtype: float64


**Reading:** `Engagement_Score` (our own composite) tops the list, which is expected since it's built from these same activity features. Among the raw features, `Total_Trans_Ct` and `Total_Ct_Chng_Q4_Q1` (transaction count and its recent change) show the strongest correlation with churn, followed by `Total_Revolving_Bal` and `Contacts_Count_12_mon`. Credit limit and age are both essentially uncorrelated (|r| < 0.03) — reinforcing that this is an **activity/engagement** story, not a demographic one.

## Summary — answers to the four EDA business questions

1. **Who churns?** Not defined by income, gender, or age — defined by *behavior*: low transaction activity, low utilization, fewer products held, more service contacts.
2. **Does inactivity matter?** Yes — churn rate rises fairly steadily as months inactive increases.
3. **Does tenure protect against churn?** Only partially — long-tenure customers still churn at meaningful rates, so "time as a customer" alone isn't a safety net.
4. **What's the strongest signal?** Transaction count and its recent trend — a customer whose activity is dropping is the clearest early-warning sign available in this data.

These four points are exactly what should anchor the Phase 4 segmentation rules and the final business recommendations.